# Modele sklearn

## Wstępna konfiguracja

### Importowanie bibliotek

In [48]:
import pandas as pd

from mlxtend.data import loadlocal_mnist

from sklearn import model_selection
from sklearn import metrics

from sklearn import linear_model
from sklearn import tree
from sklearn import ensemble
from sklearn import neural_network

### Inicjalizacja konfiguracji

In [49]:
class config:
    
    # Ścieżki do plików
    DOWNLOADED_IMAGES_PATH = '../data/t10k-images.idx3-ubyte'
    DOWNLOADED_LABELS_PATH = '../data/t10k-labels.idx1-ubyte'

    # Ogólne ustawienia projektu
    RANDOMIZE_DATA = True
    FOLDS_CNT = 5

## Konfiguracja danych

### Wczytanie danych

In [50]:
# Wczytanie danych z plików
X, y = loadlocal_mnist(
    images_path=config.DOWNLOADED_IMAGES_PATH,
    labels_path=config.DOWNLOADED_LABELS_PATH
)

# Generowanie kolumn z id pikseli
pixel_columns = [f"pixel{i}" for i in range(len(X[0]))]

# Stworzenie pandas DataFrame
df = pd.DataFrame(X, columns=pixel_columns)

# Dodanie etykiety
df["label"] = y

# Wyświetlenie danych
df

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,label
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,7
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2
9996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3
9997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
9998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,5


### Podział danych na foldy

<img src="./images/image1.png" alt="image1" width="1300"/>
<!-- ![image1](./images/image1.png) -->
<!-- ![image1](https://towardsdatascience.com/wp-content/uploads/2023/12/1N45hocCMP0u4nXLe0WuSvw.png) -->

In [51]:
# Stworzenie kolumny kfold
df["kfold"] = -1

# Podział danych na segmenty oraz ewentualnie przelosowanie danych
folds_model = model_selection.StratifiedKFold(
    n_splits=config.FOLDS_CNT,
    shuffle=config.RANDOMIZE_DATA
)
for fold, (train, test) in enumerate(folds_model.split(df, df["label"].values)):
    df.loc[test, "kfold"] = fold
    
    print(f"{fold}. {train}, {test}")

# Wyświetlenie danych
df

0. [   0    2    3 ... 9995 9996 9999], [   1    5   11 ... 9994 9997 9998]
1. [   0    1    3 ... 9994 9997 9998], [   2    9   15 ... 9995 9996 9999]
2. [   0    1    2 ... 9997 9998 9999], [   3   14   18 ... 9982 9983 9992]
3. [   1    2    3 ... 9997 9998 9999], [   0    4    6 ... 9987 9988 9989]
4. [   0    1    2 ... 9997 9998 9999], [  10   12   20 ... 9973 9976 9979]


,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,label,kfold
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,7,3
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,1
9996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,3,1
9997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,0
9998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,5,0


## Trenowanie modelu

### Dyspozytor modelu

In [52]:
class model_dispatcher:
    
    # Tablica z nazwami modeli
    model_names = [
        # "logistic_regression",
        "SGDClassifier",
        "decision_tree_gini",
        "decision_tree_entropy",
        "random_forest",
        "neural_network"
    ]

    # Słownik z nazwami modeli oraz klasami
    models = {
        "logistic_regression": linear_model.LogisticRegression(
            max_iter=6000
        ),
        "SGDClassifier": linear_model.SGDClassifier(),
        "decision_tree_gini": tree.DecisionTreeClassifier(
            criterion="gini"
        ),
        "decision_tree_entropy": tree.DecisionTreeClassifier(
            criterion="entropy"
        ),
        "random_forest": ensemble.RandomForestClassifier(),
        "neural_network": neural_network.MLPClassifier()
    }

### Trenowanie oraz testowanie modelu

In [53]:
def use_model(df_train, df_test, model, metrics_method):
    
    # Podział danych na odpowiednie zmienne
    x_train = df_train.loc[:, df_train.columns != "label"].values
    y_train = df_train.loc[:, "label"].values

    x_test = df_test.loc[:, df_test.columns != "label"].values
    y_test = df_test.loc[:, "label"].values
    
    # Trenowanie modelu
    model.fit(x_train, y_train)

    # Wykorzystanie modelu
    predicted_data = model.predict(x_test)
    
    # Obliczenie oraz zwrócenie wyników metryki
    scores = metrics_method(
        y_true=y_test,
        y_pred=predicted_data
    )
    return scores

In [54]:
# Inicjalizacja słownika z modelami oraz metrykami ich wyników
metrics_dict = {}

# Pętla przechodząca przez modele
for model_name in model_dispatcher.model_names:
    model = model_dispatcher.models[model_name]
    
    # Inicjalizacja nowego elementu w słowniku
    metrics_dict[model_name] = []

    # Testowanie modelu na różnych foldach
    for fold in range(config.FOLDS_CNT):
        
        # Inicjalizacja train oraz test dataframe
        df_train = df.loc[df["kfold"] != fold, df.columns != "kfold"]
        df_test = df.loc[df["kfold"] == fold, df.columns != "kfold"]

        # Obliczanie accuracy oraz dodanie do słownika
        score = use_model(df_train, df_test, model, metrics.accuracy_score)
        metrics_dict[model_name].append(score)
        
        print(f"{model_name}: {fold} - {score:.3f}")

SGDClassifier: 0 - 0.882
SGDClassifier: 1 - 0.876
SGDClassifier: 2 - 0.889
SGDClassifier: 3 - 0.878
SGDClassifier: 4 - 0.877
decision_tree_gini: 0 - 0.801
decision_tree_gini: 1 - 0.796
decision_tree_gini: 2 - 0.813
decision_tree_gini: 3 - 0.824
decision_tree_gini: 4 - 0.800
decision_tree_entropy: 0 - 0.808
decision_tree_entropy: 1 - 0.812
decision_tree_entropy: 2 - 0.830
decision_tree_entropy: 3 - 0.830
decision_tree_entropy: 4 - 0.806
random_forest: 0 - 0.954
random_forest: 1 - 0.949
random_forest: 2 - 0.953
random_forest: 3 - 0.950
random_forest: 4 - 0.951
neural_network: 0 - 0.919
neural_network: 1 - 0.923
neural_network: 2 - 0.925
neural_network: 3 - 0.914
neural_network: 4 - 0.914


## Metryki

### Słownik metryk

In [55]:
metrics_dict

{'SGDClassifier': [0.8825, 0.8765, 0.889, 0.878, 0.877],
 'decision_tree_gini': [0.8015, 0.7965, 0.813, 0.8235, 0.8005],
 'decision_tree_entropy': [0.808, 0.8125, 0.83, 0.8295, 0.806],
 'random_forest': [0.9535, 0.949, 0.953, 0.95, 0.9505],
 'neural_network': [0.919, 0.923, 0.9255, 0.9145, 0.9145]}

### Metryki w DataFrame-ach

In [56]:
metrics_df = pd.DataFrame(metrics_dict)

display(metrics_df)
display(metrics_df.mean().to_frame().transpose().rename(index={0: "avg"}))

,SGDClassifier,decision_tree_gini,decision_tree_entropy,random_forest,neural_network
0,0.8825,0.8015,0.8080,0.9535,0.9190
1,0.8765,0.7965,0.8125,0.9490,0.9230
2,0.8890,0.8130,0.8300,0.9530,0.9255
3,0.8780,0.8235,0.8295,0.9500,0.9145
4,0.8770,0.8005,0.8060,0.9505,0.9145


,SGDClassifier,decision_tree_gini,decision_tree_entropy,random_forest,neural_network
avg,0.8806,0.807,0.8172,0.9512,0.9193


In [57]:
metrics_df_tran = pd.DataFrame(metrics_dict).transpose()

display(metrics_df_tran)
display(metrics_df_tran.mean(axis=1).to_frame().rename(columns={0: "avg"}))

,0,1,2,3,4
SGDClassifier,0.8825,0.8765,0.8890,0.8780,0.8770
decision_tree_gini,0.8015,0.7965,0.8130,0.8235,0.8005
decision_tree_entropy,0.8080,0.8125,0.8300,0.8295,0.8060
random_forest,0.9535,0.9490,0.9530,0.9500,0.9505
neural_network,0.9190,0.9230,0.9255,0.9145,0.9145


,avg
SGDClassifier,0.8806
decision_tree_gini,0.8070
decision_tree_entropy,0.8172
random_forest,0.9512
neural_network,0.9193
